In [2]:
import pandas as pd
import numpy as np 
import scanpy as sc 
import os
import scrublet as scr 

In [3]:
# find . -type f | sort

In [4]:
study_files = {
    "Jerby-Arnon2021_10X": {
        "matrix": "Data_Jerby-Arnon2021_Sarcoma/10X/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Jerby-Arnon2021_Sarcoma/10X/Genes.txt",
        "cells":  "Data_Jerby-Arnon2021_Sarcoma/10X/Cells.csv",
        "samples": "Data_Jerby-Arnon2021_Sarcoma/Samples.csv",
        "units": "UMI"
    },
    "Jerby-Arnon2021_SmartSeq2": {
        "matrix": "Data_Jerby-Arnon2021_Sarcoma/SmartSeq2/Exp_data_TPM.mtx",
        "genes":  "Data_Jerby-Arnon2021_Sarcoma/SmartSeq2/Genes.txt",
        "cells":  "Data_Jerby-Arnon2021_Sarcoma/SmartSeq2/Cells.csv",
        "samples": "Data_Jerby-Arnon2021_Sarcoma/Samples.csv",
        "units": "TPM"
    },
    "Zhou2020": {
        "matrix": "Data_Zhou2020_Sarcoma/Exp_data_UMIcounts.mtx",
        "genes":  "Data_Zhou2020_Sarcoma/Genes.txt",
        "cells":  "Data_Zhou2020_Sarcoma/Cells.csv",
        "samples": "Data_Zhou2020_Sarcoma/Samples.csv",
        "units": "UMI"
    }
}

In [12]:
# STUDY 1

In [13]:
study_name = 'Jerby-Arnon2021_10X'
paths = study_files[study_name]

In [14]:
# matrix
adata = sc.read_mtx(paths['matrix']).T

In [15]:
genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes

In [16]:
# cell metadata
cells = pd.read_csv(paths["cells"], index_col=0)
adata.obs_names = cells.index
adata.obs_names = adata.obs_names.astype(str) 
adata.obs = adata.obs.join(cells, how="left")

In [17]:
# adding study column
adata.obs["study"] = study_name

In [18]:
f'{adata.n_obs:,} cells, {adata.n_vars:,} genes'

'9,174 cells, 9,862 genes'

In [19]:
# Drop NaN values

In [20]:
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease'] #check if present

# combine all cols that are in the study
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]

drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    
    # 1. NaN (for any dtype)
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask = drop_mask | nan_mask
    
    # 2. For object / string columns, "unknown", "NA", "nan", empty
    if vals.dtype == object:
        # Convert to string, strip and lowercase
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} values to be dropped")
            drop_mask = drop_mask | suspect

print(f"cells flagged: {drop_mask.sum()}")

# drop
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"cells remaining after drop: {adata.n_obs}")
else:
    print('no cells dropped')

cells flagged: 0
no cells dropped


In [21]:
f'{adata.n_obs:,} cells, {adata.n_vars:,} genes'

'9,174 cells, 9,862 genes'

In [22]:
# GENE COUNT FILTER (200-6000)
sc.pp.calculate_qc_metrics(adata, inplace=True)

min_genes = 200
max_genes = 6000
keep = (adata.obs["n_genes_by_counts"] >= min_genes) & \
       (adata.obs["n_genes_by_counts"] <= max_genes)
adata = adata[keep, :].copy()

In [23]:
f" {adata.n_obs:,} cells"

' 9,129 cells'

In [24]:
# DOUBLET REMOVAL USING SCRUBLET
# scrublet needs dense array, our matrix is sparse, so converting it to an array
counts = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X

In [25]:
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()

Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.50
Detected doublet rate = 0.1%
Estimated detectable doublet fraction = 40.4%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.2%
Elapsed time: 6.9 seconds


In [26]:
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets

In [27]:
adata = adata[~adata.obs["predicted_doublet"], :].copy()

In [28]:
f"   After doublet removal- {adata.n_obs:,} cells"

'   After doublet removal- 9,121 cells'

In [30]:
# list MT- cells
print('cells having mitochondrial gene: ')
print((adata[:, adata.var_names.str.startswith('MT-')].X.sum(axis=1) > 0).sum())

cells having mitochondrial gene: 
9121


In [31]:
# Dead cell removal
adata.var['mt'] = adata.var_names.str.startswith('MT-') #get mitochondrial genes
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
max_pct_mt = 15
keep_mt = adata.obs['pct_counts_mt'] <= max_pct_mt
adata = adata[keep_mt, :].copy()

In [32]:
f"after MT filter: {adata.n_obs:,} cells"

'after MT filter: 9,017 cells'

In [33]:
# Convert columns to string (conflict for h5ad)
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)

In [35]:
adata.write_h5ad('sarcoma_study1_qc.h5ad')

In [4]:
# Universal script so that it works for all upcoming studies

In [5]:
# study 2

In [9]:
study_name = 'Jerby-Arnon2021_SmartSeq2'
paths = study_files[study_name]

adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes

cells = pd.read_csv(paths["cells"], index_col=0)
adata.obs_names = cells.index
adata.obs_names = adata.obs_names.astype(str) 
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name

print('cell x gene: ')
print(f'{adata.n_obs:,} cells, {adata.n_vars:,} genes')

print('dropping NaN values')
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease'] #check if present

# combine all cols that are in the study
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]

drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    
    # 1. NaN (for any dtype)
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask = drop_mask | nan_mask
    
    # 2. For object / string columns, "unknown", "NA", "nan", empty
    if vals.dtype == object:
        # Convert to string, strip and lowercase
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} values to be dropped")
            drop_mask = drop_mask | suspect

print(f"cells flagged: {drop_mask.sum()}")

# drop
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"cells remaining after drop: {adata.n_obs}")
else:
    print('no cells dropped')


print('cells and genes after NaN drop')
print(f'{adata.n_obs:,} cells, {adata.n_vars:,} genes')


print('gene count filter\n')
# GENE COUNT FILTER (200-6000)
sc.pp.calculate_qc_metrics(adata, inplace=True)

min_genes = 200
max_genes = 6000
keep = (adata.obs["n_genes_by_counts"] >= min_genes) & \
       (adata.obs["n_genes_by_counts"] <= max_genes)
adata = adata[keep, :].copy()

print(f" {adata.n_obs:,} cells")

print('doublet removal')
# DOUBLET REMOVAL USING SCRUBLET
# scrublet needs dense array, our matrix is sparse, so converting it to an array
counts = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()

print(f"   After doublet removal- {adata.n_obs:,} cells")

# list MT- cells
print('cells having mitochondrial gene: ')
print((adata[:, adata.var_names.str.startswith('MT-')].X.sum(axis=1) > 0).sum())


# Dead cell removal
adata.var['mt'] = adata.var_names.str.startswith('MT-') #get mitochondrial genes
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
max_pct_mt = 15
keep_mt = adata.obs['pct_counts_mt'] <= max_pct_mt
adata = adata[keep_mt, :].copy()

print(f"after MT filter: {adata.n_obs:,} cells")


# Convert columns to string (conflict for h5ad)
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)


adata.write_h5ad('sarcoma_study2_qc.h5ad')

cell x gene: 
6,951 cells, 23,686 genes
dropping NaN values
cells flagged: 0
no cells dropped
cells and genes after NaN drop
6,951 cells, 23,686 genes
gene count filter

 6,221 cells
doublet removal
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.52
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 46.2%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.1%
Elapsed time: 7.6 seconds
   After doublet removal- 6,218 cells
cells having mitochondrial gene: 
0
after MT filter: 6,218 cells


In [ ]:
# study 3

In [10]:
study_name = 'Zhou2020'
paths = study_files[study_name]

adata = sc.read_mtx(paths['matrix']).T

genes = pd.read_csv(paths["genes"], header=None, sep="\t").iloc[:, 0].tolist()
adata.var_names = genes
adata.var_names_make_unique()  #only for this study as var has duplicates


cells = pd.read_csv(paths["cells"], index_col=0)
adata.obs_names = cells.index
adata.obs_names = adata.obs_names.astype(str) 
adata.obs = adata.obs.join(cells, how="left")

adata.obs["study"] = study_name

print('cell x gene: ')
print(f'{adata.n_obs:,} cells, {adata.n_vars:,} genes')

print('dropping NaN values')
CRITICAL_COLS = ['cell_type', 'sample']
EXTRA_COLS = ['cancer_type', 'disease'] #check if present

# combine all cols that are in the study
all_cols_to_check = [c for c in CRITICAL_COLS + EXTRA_COLS if c in adata.obs.columns]

drop_mask = pd.Series(False, index=adata.obs.index)

for col in all_cols_to_check:
    vals = adata.obs[col]
    
    # 1. NaN (for any dtype)
    nan_mask = vals.isna()
    if nan_mask.any():
        print(f"   {col}: {nan_mask.sum()} NaN")
        drop_mask = drop_mask | nan_mask
    
    # 2. For object / string columns, "unknown", "NA", "nan", empty
    if vals.dtype == object:
        # Convert to string, strip and lowercase
        str_vals = vals.astype(str).str.strip().str.lower()
        suspect = str_vals.isin(['unknown', 'na', 'nan', 'none', ''])
        if suspect.any():
            print(f"   {col}: {suspect.sum()} values to be dropped")
            drop_mask = drop_mask | suspect

print(f"cells flagged: {drop_mask.sum()}")

# drop
if drop_mask.any():
    adata = adata[~drop_mask, :].copy()
    print(f"cells remaining after drop: {adata.n_obs}")
else:
    print('no cells dropped')


print('cells and genes after NaN drop')
print(f'{adata.n_obs:,} cells, {adata.n_vars:,} genes')


print('gene count filter\n')
# GENE COUNT FILTER (200-6000)
sc.pp.calculate_qc_metrics(adata, inplace=True)

min_genes = 200
max_genes = 6000
keep = (adata.obs["n_genes_by_counts"] >= min_genes) & \
       (adata.obs["n_genes_by_counts"] <= max_genes)
adata = adata[keep, :].copy()

print(f" {adata.n_obs:,} cells")

print('doublet removal')
# DOUBLET REMOVAL USING SCRUBLET
# scrublet needs dense array, our matrix is sparse, so converting it to an array
counts = adata.X.toarray() if hasattr(adata.X, "toarray") else adata.X
scrub = scr.Scrublet(counts, expected_doublet_rate=0.05)
doublet_scores, predicted_doublets = scrub.scrub_doublets()
adata.obs["doublet_score"] = doublet_scores
adata.obs["predicted_doublet"] = predicted_doublets
adata = adata[~adata.obs["predicted_doublet"], :].copy()

print(f"   After doublet removal- {adata.n_obs:,} cells")

# list MT- cells
print('cells having mitochondrial gene: ')
print((adata[:, adata.var_names.str.startswith('MT-')].X.sum(axis=1) > 0).sum())


# Dead cell removal
adata.var['mt'] = adata.var_names.str.startswith('MT-') #get mitochondrial genes
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], inplace=True)
max_pct_mt = 15
keep_mt = adata.obs['pct_counts_mt'] <= max_pct_mt
adata = adata[keep_mt, :].copy()

print(f"after MT filter: {adata.n_obs:,} cells")


# Convert columns to string (conflict for h5ad)
for col in adata.obs.columns:
    if adata.obs[col].dtype == 'object':
        adata.obs[col] = adata.obs[col].astype(str)


adata.write_h5ad('sarcoma_study3_qc.h5ad')

cell x gene: 
64,557 cells, 32,864 genes
dropping NaN values
cells flagged: 0
no cells dropped
cells and genes after NaN drop
64,557 cells, 32,864 genes
gene count filter

 63,195 cells
doublet removal
Preprocessing...
Simulating doublets...
Embedding transcriptomes using PCA...
Calculating doublet scores...
Automatically set threshold at doublet score = 0.73
Detected doublet rate = 0.0%
Estimated detectable doublet fraction = 7.5%
Overall doublet rate:
	Expected   = 5.0%
	Estimated  = 0.1%
Elapsed time: 131.2 seconds
   After doublet removal- 63,190 cells
cells having mitochondrial gene: 
63190
after MT filter: 63,190 cells


In [ ]:
# todo: normalization

In [1]:
import glob

In [5]:
files = {
    "sarcoma_study1_qc.h5ad": "Jerby-Arnon2021_10X",
    "sarcoma_study2_qc.h5ad": "Jerby-Arnon2021_SmartSeq2",
    "sarcoma_study3_qc.h5ad": "Zhou2020"
}

for fname, study_name in files.items():
    unit = study_files[study_name]["units"]
    adata = sc.read_h5ad(fname)

    if unit == "TPM":
        sc.pp.log1p(adata)
    else:
        sc.pp.normalize_total(adata, target_sum=1e6)
        sc.pp.log1p(adata)

    out_name = f"{study_name}_normalized.h5ad"
    adata.write_h5ad(out_name)
    print(f"{study_name}: normalised ({unit}) – max value {adata.X.max():.2f}")

Jerby-Arnon2021_10X: normalised (UMI) – max value 12.72
Jerby-Arnon2021_SmartSeq2: normalised (TPM) – max value 10.71
Zhou2020: normalised (UMI) – max value 13.24
